# HTML Utilities API Reference

Developer-facing statements defined in `libs/core/langchain_core/utils/html.py`.

# `PREFIXES_TO_IGNORE`

Link prefixes excluded by the default link-matching pattern.

```python
PREFIXES_TO_IGNORE = ("javascript:", "mailto:", "#")
```

---

# `SUFFIXES_TO_IGNORE`

File suffixes excluded by the default link-matching pattern.

```python
SUFFIXES_TO_IGNORE = (
    ".css",
    ".js",
    ".ico",
    ".png",
    ".jpg",
    ".jpeg",
    ".gif",
    ".svg",
    ".csv",
    ".bz2",
    ".zip",
    ".epub",
    ".webp",
    ".pdf",
    ".docx",
    ".xlsx",
    ".pptx",
    ".pptm",
)
```

---

# `SUFFIXES_TO_IGNORE_REGEX`

Negative-lookahead expression generated from `SUFFIXES_TO_IGNORE`.

```python
SUFFIXES_TO_IGNORE_REGEX = (
    "(?!" + "|".join([re.escape(s) + r"[\#'\"]" for s in SUFFIXES_TO_IGNORE]) + ")"
)
```

---

# `PREFIXES_TO_IGNORE_REGEX`

Negative-lookahead expression generated from `PREFIXES_TO_IGNORE`.

```python
PREFIXES_TO_IGNORE_REGEX = (
    "(?!" + "|".join([re.escape(s) for s in PREFIXES_TO_IGNORE]) + ")"
)
```

---

# `DEFAULT_LINK_REGEX`

Default regular expression used to extract eligible `href` values.

```python
DEFAULT_LINK_REGEX = (
    rf"href=[\"']{PREFIXES_TO_IGNORE_REGEX}((?:{SUFFIXES_TO_IGNORE_REGEX}.)*?)[\#'\"]"
)
```

---

# `find_all_links`

Extracts unique links from raw HTML.

```python
find_all_links(
    raw_html: str, # Raw HTML to search
    *,
    pattern: str | re.Pattern[str] | None = None, # Extraction regex, or None to use DEFAULT_LINK_REGEX
) -> list[str] # Unique extracted links
```

The function applies `re.findall` and removes duplicates through a set. Result ordering is not guaranteed.

---

# `extract_sub_links`

Extracts links from raw HTML, converts them to absolute URLs, and applies optional scope filtering.

```python
extract_sub_links(
    raw_html: str, # Raw HTML to search
    url: str, # URL of the HTML document and base for resolving relative links
    *,
    base_url: str | None = None, # URL used to determine whether resolved links are inside the allowed scope
    pattern: str | re.Pattern[str] | None = None, # Extraction regex forwarded to find_all_links
    prevent_outside: bool = True, # Whether to exclude links outside base_url
    exclude_prefixes: Sequence[str] = (), # Absolute URL prefixes to exclude
    continue_on_failure: bool = False, # Whether to log and skip links that fail during processing
) -> list[str] # Filtered absolute URLs
```

When `base_url` is `None`, `url` is used as the allowed base. Absolute HTTP and HTTPS links are preserved, protocol-relative links inherit the document URL's scheme, and other links are resolved with `urljoin`. Query strings are retained.

Resolved URLs are deduplicated before filtering. URLs matching `exclude_prefixes` are removed first. When `prevent_outside` is `True`, a URL must have the same network location as the base URL and begin with the complete base URL string. Result ordering is not guaranteed.

When processing a link raises an exception, the exception is re-raised unless `continue_on_failure=True`, in which case a warning is logged and that link is skipped.

In [ ]:
from langchain_core.utils.html import extract_sub_links, find_all_links # Import the HTML link utilities


raw_html = """ # Create sample HTML containing different link types
<html> # Open the HTML document
    <body> # Open the document body
        <a href="guide.html">Guide</a> # Add a relative link
        <a href="/docs/api">API</a> # Add a root-relative link
        <a href="https://example.com/docs/tutorial">Tutorial</a> # Add an absolute internal link
        <a href="https://example.com/docs/private/admin">Private</a> # Add a link that will be excluded
        <a href="https://external.com/article">External</a> # Add an external link
        <a href="mailto:support@example.com">Email</a> # Add an ignored email link
        <a href="javascript:void(0)">Click</a> # Add an ignored JavaScript link
        <a href="image.png">Image</a> # Add an ignored image link
        <a href="manual.pdf">Manual</a> # Add an ignored PDF link
        <a href="guide.html#installation">Installation</a> # Add a link containing a fragment
    </body> # Close the document body
</html> # Close the HTML document
""" # Finish the sample HTML string

In [ ]:
# Extract eligible links without resolving them

links = find_all_links(raw_html) # Extract unique links using the default filtering pattern

for link in sorted(links): # Sort the links because the function does not guarantee ordering
    print(link) # Display each extracted link


In [ ]:
# Convert links into filtered absolute URLs
page_url = "https://example.com/docs/index.html" # Specify the URL of the current HTML page
base_url = "https://example.com/docs/" # Restrict accepted links to this documentation section

sub_links = extract_sub_links( # Extract and normalize links from the HTML
    raw_html=raw_html, # Provide the HTML content
    url=page_url, # Use the current page URL to resolve relative links
    base_url=base_url, # Restrict links to the documentation base URL
    prevent_outside=True, # Remove external links and links outside the base path
    exclude_prefixes=("https://example.com/docs/private",), # Exclude private documentation URLs
) # Finish extracting the links

for link in sorted(sub_links): # Sort the result because output ordering is not guaranteed
    print(link) # Display each accepted absolute URL